# 그릇 밀어서 정렬하기 — MuJoCo + SB3 PPO (Colab 뼈대)

**아직 실행하지 않은 뼈대입니다.** 런타임: Colab (CPU 로도 수 시간, GPU 불필요).

- 목표: 테이블 위 그릇을 *잡지 않고* 밀어서 목표 위치(사용자 앞)로 보낸다.
- 행동은 **고수준 밀기 1번** = `[접근 각도, 밀기 방향 오프셋, 밀기 거리]` → 실제 로봇에서는 기존 `movel` 저속 직선 이동 3개로 실행
  (관절 제어가 아니므로 sim2real 차이가 작고, 테이블 위 저속이라 안전).
- 관찰: `[그릇 x, y, cosθ, sinθ, 목표 x, y]` (테이블 좌표 m). 실제에서는 YOLO + table_view 로 얻음.
- 도메인 랜덤화: 마찰, 질량, 그릇 크기, 밀기 지점 오차.
- 결과: `push_policy.onnx` → 노트북 CPU 에서 onnxruntime 으로 추론 (assist_arm 에 서브태스크로 붙일 예정).

In [ ]:
!pip -q install mujoco gymnasium stable-baselines3 onnx onnxruntime
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/assist_arm/rl_push'
import os; os.makedirs(OUT, exist_ok=True)

## 1. MuJoCo 장면: 테이블 + 그릇(원통) + 밀대(mocap 구)

In [ ]:
import numpy as np, mujoco

def make_xml(radius=0.07, height=0.04, mass=0.3, friction=0.6):
    return f"""
<mujoco>
  <option timestep="0.002" gravity="0 0 -9.81"/>
  <worldbody>
    <geom name="table" type="plane" size="0.6 0.6 0.01" friction="{friction} 0.005 0.0001"/>
    <body name="dish" pos="0 0 {height/2}">
      <freejoint/>
      <geom name="dish_g" type="cylinder" size="{radius} {height/2}" mass="{mass}" friction="{friction} 0.005 0.0001"/>
    </body>
    <body name="pusher" mocap="true" pos="0.3 0 0.02">
      <geom type="sphere" size="0.015" rgba="1 0 0 1" friction="0.3 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>"""

## 2. Gymnasium 환경: 한 step = 밀기 한 번

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class PushDishEnv(gym.Env):
    """action: [접근각 a∈[-1,1]→±π, 밀기 방향 오프셋 o∈[-1,1]→±30°, 거리 d∈[-1,1]→0.02~0.15 m]"""
    def __init__(self, max_pushes=5, tol=0.02, randomize=True):
        self.max_pushes, self.tol, self.randomize = max_pushes, tol, randomize
        self.action_space = spaces.Box(-1, 1, (3,), np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, (6,), np.float32)

    def _build(self):
        r = self.np_random
        kw = dict(radius=r.uniform(0.05, 0.09), mass=r.uniform(0.1, 0.6), friction=r.uniform(0.3, 0.9)) \
            if self.randomize else {}
        self.model = mujoco.MjModel.from_xml_string(make_xml(**kw))
        self.data = mujoco.MjData(self.model)
        self.radius = kw.get('radius', 0.07)

    def _obs(self):
        x, y = self.data.qpos[0:2]
        q = self.data.qpos[3:7]
        yaw = np.arctan2(2 * (q[0] * q[3] + q[1] * q[2]), 1 - 2 * (q[2] ** 2 + q[3] ** 2))
        return np.array([x, y, np.cos(yaw), np.sin(yaw), *self.goal], np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._build()
        self.data.qpos[0:2] = self.np_random.uniform(-0.15, 0.15, 2)
        self.goal = self.np_random.uniform(-0.25, 0.25, 2)
        mujoco.mj_forward(self.model, self.data)
        self.k = 0
        return self._obs(), {}

    def _move_pusher(self, p0, p1, speed=0.05):
        n = max(2, int(np.linalg.norm(p1 - p0) / speed / self.model.opt.timestep))
        for t in np.linspace(0, 1, n):
            self.data.mocap_pos[0] = [*(p0 + t * (p1 - p0)), 0.02]
            mujoco.mj_step(self.model, self.data)

    def step(self, action):
        a, o, d = np.clip(action, -1, 1)
        dish = self.data.qpos[0:2].copy()
        ang = a * np.pi
        dist = 0.02 + (d + 1) / 2 * 0.13
        start = dish + (self.radius + 0.04) * np.array([np.cos(ang), np.sin(ang)])
        start += self.np_random.normal(0, 0.005, 2) if self.randomize else 0       # 실제 위치 오차 흉내
        push_dir = -np.array([np.cos(ang + o * np.pi / 6), np.sin(ang + o * np.pi / 6)])
        before = np.linalg.norm(dish - self.goal)
        self.data.mocap_pos[0] = [*start, 0.06]                       # 위에서 내려와서
        self._move_pusher(start, start)
        self._move_pusher(start, start + push_dir * (0.04 + dist))     # 밀고
        self._move_pusher(self.data.mocap_pos[0][:2].copy(), self.data.mocap_pos[0][:2] - push_dir * 0.03)  # 살짝 빠짐
        for _ in range(300):
            mujoco.mj_step(self.model, self.data)                     # 멈출 때까지
        after = np.linalg.norm(self.data.qpos[0:2] - self.goal)
        self.k += 1
        off_table = np.any(np.abs(self.data.qpos[0:2]) > 0.55)
        success = after < self.tol
        reward = (before - after) * 10 - 0.05 + (5.0 if success else 0.0) - (5.0 if off_table else 0.0)
        done = success or off_table
        return self._obs(), float(reward), done, self.k >= self.max_pushes, {'success': success}

## 3. 확인 (랜덤 행동)

In [ ]:
env = PushDishEnv()
obs, _ = env.reset(seed=0)
for _ in range(5):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    print(np.round(obs, 3), round(r, 3), info)
    if term or trunc: break

## 4. PPO 학습 (병렬 환경 8개)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CheckpointCallback

venv = make_vec_env(PushDishEnv, n_envs=8, seed=0)
model = PPO('MlpPolicy', venv, n_steps=256, batch_size=512, gamma=0.95, learning_rate=3e-4,
            policy_kwargs=dict(net_arch=[64, 64]), verbose=1)
model.learn(total_timesteps=300_000, callback=CheckpointCallback(50_000, OUT, name_prefix='ppo_push'))
model.save(f'{OUT}/ppo_push_final')

## 5. 평가: 성공률 / 평균 밀기 횟수 (랜덤화 켠 상태)

In [ ]:
def evaluate(model, n=200):
    env, ok, pushes = PushDishEnv(), 0, []
    for i in range(n):
        obs, _ = env.reset(seed=1000 + i)
        for k in range(env.max_pushes):
            act, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, info = env.step(act)
            if term or trunc: break
        ok += info['success']; pushes.append(k + 1)
    print(f'성공률 {ok / n:.2%}, 평균 밀기 {np.mean(pushes):.2f}회')
evaluate(model)

## 6. ONNX 내보내기 → 노트북(onnxruntime CPU)에서 추론

In [ ]:
import torch
class Actor(torch.nn.Module):
    def __init__(self, policy):
        super().__init__(); self.p = policy
    def forward(self, obs):
        return self.p.get_distribution(obs).distribution.mean.clamp(-1, 1)   # 결정적 행동
actor = Actor(model.policy).eval()
torch.onnx.export(actor, torch.zeros(1, 6), f'{OUT}/push_policy.onnx', input_names=['obs'], output_names=['action'],
                  opset_version=17)
import onnxruntime as ort
s = ort.InferenceSession(f'{OUT}/push_policy.onnx')
print(s.run(None, {'obs': np.zeros((1, 6), np.float32)}))

## 7. 실제 로봇에 붙일 때 (메모)
- 좌표: 관찰 x, y 는 테이블 원점 기준 m → `table_layout.yaml` 에 테이블 원점(베이스 mm) 추가 후 변환.
- 행동 → 로봇: `start` 위 60mm 로 이동 → 테이블+20mm 로 하강 → `push_dir` 로 `dist` 만큼 저속(≤40 mm/s) movel → 30mm 빠짐 → 다시 YOLO 관찰.
- 그리퍼는 닫은 상태로 손가락 끝을 밀대로 사용. 그릇이 손 20cm 안이면 safety_monitor 가 일시정지.
- 먼저 sim 성공률 90%+ 확인, 실제에서는 빈 플라스틱 그릇으로만.